# ЛР-3: Structured Streaming

Среда: **Google Colab**.

Полное методическое описание, критерии оценивания и `TEACH CARD` находятся в одноимённом `.md` файле комплекта.

Сквозной пайплайн курса:

$$
\text{Sense} \rightarrow \text{Collect} \rightarrow \text{Stream} \rightarrow
\text{Store} \rightarrow \text{Process} \rightarrow \text{Learn} \rightarrow \text{Teach}
$$


In [ ]:
!pip -q install "pyspark==4.2.0"

from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab03_StructuredStreaming")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Colab-safe источник неограниченного потока.
# Он заменяет Kafka broker в обязательной части лабораторной,
# сохраняя семантику unbounded stream + event time + windows.
source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 30)
    .option("numPartitions", 2)
    .load()
)

telemetry = (
    source
    .withColumn("robot_id", F.concat(F.lit("robot-"), (F.col("value") % 4).cast("string")))
    .withColumn(
        "event_time",
        F.when(
            F.col("value") % 17 == 0,
            F.col("timestamp") - F.expr("INTERVAL 20 SECONDS")
        ).otherwise(F.col("timestamp"))
    )
    .withColumn("motor_temp", F.lit(43.0) + (F.col("value") % 25).cast("double") * 0.55)
    .withColumn("vibration", F.abs(F.sin(F.col("value") / 8.0)))
    .select("robot_id", "event_time", "motor_temp", "vibration")
)

windowed = (
    telemetry
    .withWatermark("event_time", "10 seconds")
    .groupBy(
        F.window("event_time", "10 seconds", "5 seconds"),
        "robot_id"
    )
    .agg(
        F.count("*").alias("events"),
        F.avg("motor_temp").alias("avg_temp"),
        F.max("motor_temp").alias("max_temp"),
        F.sqrt(F.avg(F.pow("vibration", 2))).alias("vibration_rms")
    )
)

query = (
    windowed.writeStream
    .format("memory")
    .queryName("robot_windows")
    .outputMode("update")
    .trigger(processingTime="1 second")
    .start()
)

# Накопить несколько micro-batch и корректно остановить query.
time.sleep(12)
query.processAllAvailable()
query.stop()

result = spark.sql("""
SELECT
    window.start AS window_start,
    window.end AS window_end,
    robot_id,
    events,
    ROUND(avg_temp, 2) AS avg_temp,
    ROUND(max_temp, 2) AS max_temp,
    ROUND(vibration_rms, 4) AS vibration_rms
FROM robot_windows
ORDER BY window_start DESC, robot_id
""")

result.show(40, truncate=False)
print("Rows in materialized streaming result:", result.count())

# Техническое задание: изменить watermark 10s -> 30s и сравнить
# число сохранённых опоздавших событий.


## TEACH CARD

После выполнения кода заполните `TEACH CARD` из `.md`-файла лабораторной работы и приложите его к отчёту.
